## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Install pre-built GPU wheel while forcing compatible dependency versions
!pip install \
  llama-cpp-python==0.3.16 \
  numpy==1.26.4 \
  requests==2.32.4 \
  numba==0.60.0 \
  tensorflow==2.19.0 \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 \
  --force-reinstall --no-cache-dir

**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [ ]:
# Updated for NumPy 2.x Compatibility
!pip install \
  numpy==2.0.2 \
  tensorflow>=2.18.0 \
  numba>=0.60.0 \
  huggingface_hub==0.35.3 \
  pandas>=2.2.2 \
  tiktoken==0.12.0 \
  pymupdf==1.26.5 \
  langchain==0.3.27 \
  langchain-community==0.3.31 \
  chromadb==1.1.1 \
  sentence-transformers==5.1.1 \
  llama-cpp-python==0.3.16 \
  scipy>=1.13.0\
  scikit-learn>=1.5.0\
  --no-cache-dir -q


**Note**:
- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in ***this notebook***.

In [2]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

## Question Answering using LLM

#### Downloading and Loading the model

In [3]:
# Defining the Hugging Face repository and model version for Mistral-7B fine-tuned for instruction-following
model_name_or_path = 'TheBloke/Mistral-7B-Instruct-v0.2-GGUF'

# Specifying the file name for the quantized Mistral-7B model in GGUF format (Q6_K for optimal performance)
model_basename = 'mistral-7b-instruct-v0.2.Q6_K.gguf'

In [4]:
# Downloading the specified model file from Hugging Face Hub and store its local path
model_path = hf_hub_download(
    repo_id=model_name_or_path, #The Hugging Face repository containing the model
    filename=model_basename  # The specific model file to download (in GGUF format)
)
#The GGUF format is used because it provides memory-efficient storage and faster inference while maintaining compatibility across different hardware platforms.

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
#These parameters are well suited for a Google Colab CPU runtime, as 'n_ctx=2048' provides a sufficient context window for most prompts,
#'n_threads=2' aligns with the typical Colab CPU core availability for efficient parallel processing, 'n_batch=128' balances token processing speed with memory usage,
#and 'use_mlock=False' avoids memory locking issues in restricted environments like Colab.

llm = Llama(
    model_path=model_path,
    n_ctx=4096,
    n_gpu_layers=32,
    n_batch=256,
    use_mlock=False
)

ggml_cuda_init: GGML_CUDA_FORCE_MMQ:    yes
ggml_cuda_init: GGML_CUDA_FORCE_CUBLAS: no
ggml_cuda_init: found 1 CUDA devices:
  Device 0: Tesla T4, compute capability 7.5, VMM: yes
llama_model_load_from_file_impl: using device CUDA0 (Tesla T4) - 14807 MiB free
llama_model_loader: loaded meta data with 24 key-value pairs and 291 tensors from /root/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q6_K.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = mistralai_mistral-7b-instruct-v0.2
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                  

#### Response

In [6]:
#The function sends the user’s query to the Llama model with controlled generation settings ('temperature', 'top_p', 'top_k', and 'max_tokens') to produce a focused and coherent response.
#It also prevents the prompt from being repeated ('echo=False') stops generation at a defined token and returns a clean response by extracting and trimming the generated text.

def response(query, max_tokens=1024, temperature=0.2, top_p=0.9, top_k=40):

    model_output = llm(
        prompt=query,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        echo=False,
        stop=["Question:", "\n\n\n"]  # better stop signals
    )

    return model_output["choices"][0]["text"].strip()


### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [7]:
user_input_1 = "'What is the protocol for managing sepsis in a critical care unit?'"
response(user_input_1)

llama_perf_context_print:        load time =     488.89 ms
llama_perf_context_print: prompt eval time =     488.51 ms /    17 tokens (   28.74 ms per token,    34.80 tokens per second)
llama_perf_context_print:        eval time =   46788.61 ms /   959 runs   (   48.79 ms per token,    20.50 tokens per second)
llama_perf_context_print:       total time =   48711.46 ms /   976 tokens
llama_perf_context_print:    graphs reused =        928


"is a question that is frequently asked by medical students, nurses, and other healthcare professionals. Sepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. In this article, we will discuss the current guidelines for managing sepsis in a critical care unit.\n\n1. Early Recognition: The first step in managing sepsis is early recognition. Sepsis can present with non-specific symptoms such as fever, tachycardia, tachypnea, altered mental status, and lactic acidosis. The Sequential Organ Failure Assessment (SOFA) score and the Quick Sequential Organ Failure Assessment (qSOFA) score are commonly used tools for identifying patients at risk for sepsis. Patients with suspected infection and two or more qSOFA criteria or a SOFA score ≥2 should be evaluated for sepsis.\n2. Fluid Resuscitation: Once sepsis is suspected, prompt fluid resuscitation is essential. The goal is to maintain adequat

### Observations

* The response is getting truncated mid-sentence, indicating it is hitting the max_tokens limit.

* The answer starts structured but loses formatting consistency (e.g., \n1., \n2. inline instead of clean bullets).

* The response generated by the LLM is detailed and medically appropriate regarding the management of sepsis in a critical care unit.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [8]:
user_input_2 = "'What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?'"
response(user_input_2)

Llama.generate: 3 prefix-match hit, remaining 32 prompt tokens to eval
llama_perf_context_print:        load time =     488.89 ms
llama_perf_context_print: prompt eval time =     112.16 ms /    32 tokens (    3.50 ms per token,   285.32 tokens per second)
llama_perf_context_print:        eval time =   25802.08 ms /   503 runs   (   51.30 ms per token,    19.49 tokens per second)
llama_perf_context_print:       total time =   26495.92 ms /   535 tokens
llama_perf_context_print:    graphs reused =        487


"Appendicitis is a medical condition characterized by inflammation of the appendix, a small, tube-shaped organ located in the lower right side of the abdomen. The symptoms of appendicitis can vary from person to person, but some common signs include:\n\n1. Abdominal pain: The pain is typically located in the lower right side of the abdomen and may be constant or come and go. It may be mild at first but can become severe over time.\n2. Loss of appetite: People with appendicitis may lose their appetite and feel nauseous or vomit.\n3. Fever: A fever of 100.4°F (38°C) or higher is common in people with appendicitis.\n4. Diarrhea or constipation: Some people with appendicitis may experience diarrhea, while others may have constipation.\n5. Abdominal swelling: The abdomen may become swollen and tender to the touch.\n6. Inability to pass gas: People with appendicitis may have difficulty passing gas, which can contribute to abdominal pain and bloating.\n\nAppendicitis cannot be cured with medi

### Observations

* The answer again gets truncated mid-sentence, confirming a consistent token limit issue.

* Response starts with a long definition of appendicitis, which is not strictly needed → token inefficiency.

* The model does not fully answer all parts of the question:

    * Symptoms → partially covered

    * Medicine vs cure → partially covered

    * Surgical procedure → missing due to truncation

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [9]:
user_input_3 = "'What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?'"
response(user_input_3)

Llama.generate: 5 prefix-match hit, remaining 34 prompt tokens to eval
llama_perf_context_print:        load time =     488.89 ms
llama_perf_context_print: prompt eval time =      80.98 ms /    34 tokens (    2.38 ms per token,   419.84 tokens per second)
llama_perf_context_print:        eval time =   41062.22 ms /   764 runs   (   53.75 ms per token,    18.61 tokens per second)
llama_perf_context_print:       total time =   42105.40 ms /   798 tokens
llama_perf_context_print:    graphs reused =        739


"Sudden patchy hair loss, also known as alopecia areata, is a common hair disorder that affects both men and women. It is characterized by round or oval bald spots on the scalp, which can range from a few hairs to large areas of hair loss. The exact cause of alopecia areata is not known, but it is believed to be an autoimmune condition, where the body's immune system attacks the hair follicles.\n\nThere are several treatments and solutions for addressing sudden patchy hair loss:\n\n1. Corticosteroids: Corticosteroids are anti-inflammatory medications that can help reduce inflammation and suppress the immune system's attack on the hair follicles. They can be applied topically or taken orally. Topical corticosteroids are the most commonly used treatment for alopecia areata.\n2. Immunotherapies: Immunotherapies, such as minoxidil and anthralin, can help stimulate hair growth and reduce inflammation. Minoxidil is a topical medication that is applied directly to the scalp, while anthralin i

### Observations

* Response is again truncated mid-treatment explanation, indicating persistent token limit issues.

* The answer starts structured but loses formatting consistency (e.g., \n1., \n2. inline instead of clean bullets).

* The question asks for:

    * Treatments/solutions  (partially covered)

    * Causes (partially covered)

    However, neither section is fully completed or clearly separated.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [10]:
user_input_4 = "'What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?'"
response(user_input_4)

Llama.generate: 3 prefix-match hit, remaining 28 prompt tokens to eval
llama_perf_context_print:        load time =     488.89 ms
llama_perf_context_print: prompt eval time =      76.12 ms /    28 tokens (    2.72 ms per token,   367.84 tokens per second)
llama_perf_context_print:        eval time =   30484.38 ms /   591 runs   (   51.58 ms per token,    19.39 tokens per second)
llama_perf_context_print:       total time =   31203.74 ms /   619 tokens
llama_perf_context_print:    graphs reused =        571


"A person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function, is typically diagnosed with a traumatic brain injury (TBI). The recommended treatments for TBI depend on the severity and location of the injury, as well as the individual's age, overall health, and specific symptoms.\n\nHere are some common treatments for TBI:\n\n1. Emergency care: If the injury is severe, the first priority is to provide emergency care to stabilize the person's condition and prevent further damage. This may include surgery to remove hematomas or other debris, administering oxygen, and managing blood pressure and other vital signs.\n2. Medications: Depending on the symptoms, various medications may be prescribed to manage conditions such as swelling, pain, seizures, or infections. For example, corticosteroids may be used to reduce swelling, while anticonvulsants may be prescribed to prevent seizures.\n3. Rehabilitation: Rehabilitation is an 

### Observations

* Response is again cut off mid-sentence, confirming persistent truncation.

* Question asks for recommended treatments, and the model starts covering:

    * Medical management

    * Rehabilitation

    * Medications (incomplete due to cutoff)

* Missing additional important areas like:

    * Long-term care

    * Psychological support

    * Prognosis considerations

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [11]:
user_input_5 = "'What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?'"
response(user_input_5)

Llama.generate: 3 prefix-match hit, remaining 35 prompt tokens to eval
llama_perf_context_print:        load time =     488.89 ms
llama_perf_context_print: prompt eval time =      79.38 ms /    35 tokens (    2.27 ms per token,   440.90 tokens per second)
llama_perf_context_print:        eval time =   25556.26 ms /   489 runs   (   52.26 ms per token,    19.13 tokens per second)
llama_perf_context_print:       total time =   26109.26 ms /   524 tokens
llama_perf_context_print:    graphs reused =        473


"A fractured leg, also known as a broken leg, is a common injury that can occur during hiking due to falls, tripping over rocks, or other accidents. Here are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip:\n\n1. Assess the situation: If someone has fractured their leg during a hiking trip, the first step is to assess the situation. Check if the person is in pain, if they can move, and if there are any other injuries. If the person is in severe pain, unable to move, or if there are signs of shock (such as pale skin, rapid heartbeat, or shallow breathing), call for emergency medical help immediately.\n2. Provide first aid: If the person is stable and able to move, provide first aid. This may include immobilizing the leg with a splint or a makeshift sling made from a shirt or other material. Be sure to keep the leg elevated to reduce swelling and pain.\n3. Get to safety: If possible, help the person get to a safe and sheltered a

### Observations

#### Overall Observations

### **Strengths and Limitations of Mistral 7B in This Scenario**

### **Strengths**

* **Comprehensive Coverage:** The Mistral 7B model generates detailed, medically sound responses that span a wide range of relevant concepts and considerations.

* **Structured Output:** Responses are generally well-organized, often leveraging numbered lists or logical sequencing, making them easier to follow and interpret.

* **Contextual Relevance:** The model demonstrates a strong ability to understand the query and provide contextually appropriate information, addressing multiple facets of the question.

---

### **Limitations**

* **Response Truncation:** A consistent issue across outputs is that responses are cut off mid-sentence, particularly for multi-part or complex queries. This strongly indicates token limit constraints, resulting in incomplete answers and reduced usability.

* **Overly Generic Guidance:** While medically accurate, some responses remain high-level and generic (e.g., standard treatment protocols), lacking the depth or specificity required for nuanced scenarios.

* **Limited Contextual Depth:** The model tends to prioritize broad explanations over tailored insights, making outputs feel more like general guidelines rather than scenario-specific recommendations.

---

### **Next Steps**

To address these limitations, we will apply **prompt engineering techniques** to:

* Enforce concise, structured responses
* Improve completeness within token limits
* Drive more specific and context-aware outputs


## Question Answering using LLM with Prompt Engineering

In [21]:
# Basic parameters
def generate_llama_response(
    query,
    instruction = """
Answer in sections:
1. Key Points
2. Details
3. Conclusion
Keep it concise.
""",
    max_tokens=1024,
    temperature=0.4,
    top_p=0.9,
    top_k=40,
    repeat_penalty=1.1,
):
    # Proper LLaMA instruction format
    prompt = f"""<s>[INST] <<SYS>>
{instruction}
<</SYS>>

{query} [/INST]"""

    response = llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repeat_penalty=repeat_penalty,
        stop=["</s>", "[INST]"],  # stronger stop signals
        echo=False,
    )

    text = response["choices"][0]["text"].strip()

    # Basic cleanup (fix escaped newlines)
    text = text.replace("\\n", "\n")

    return text


### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [46]:
generate_llama_response('What is the protocol for managing sepsis in a critical care unit?')



Llama.generate: 61 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     488.89 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   32516.92 ms /   600 runs   (   54.19 ms per token,    18.45 tokens per second)
llama_perf_context_print:       total time =   33795.48 ms /   601 tokens
llama_perf_context_print:    graphs reused =        580


'1. Key Points:\n- Early recognition and initiation of sepsis treatment are crucial to improve outcomes.\n- ABCDE approach (Airway, Breathing, Circulation, Disability, Exposure) is used for initial assessment and management.\n- Antibiotics should be administered as soon as possible based on clinical suspicion and microbiological results.\n- Fluid resuscitation is essential to maintain adequate tissue perfusion.\n- Vasopressors may be required to maintain blood pressure and organ perfusion.\n- Corticosteroids, vasoactive agents, and other adjunctive therapies may be considered in certain situations.\n- Continuous monitoring, including hemodynamic support, oxygenation, and close observation for organ dysfunction, is essential.\n\n2. Details:\n\na) Early recognition: Sepsis should be suspected when a patient presents with infection and signs of organ dysfunction. The Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score can help identify organ dysfunction.\n\nb) Initial assess

In [14]:
# Modified parameters - Temperature = 0.0
def generate_llama_response_temparature(
    query,
    instruction = """
Answer in sections:
1. Key Points
2. Details
3. Conclusion
Keep it concise.
""",
    max_tokens=1024,
    temperature=0.0,
    top_p=0.9,
    top_k=40,
    repeat_penalty=1.1,
):
    # Proper LLaMA instruction format
    prompt = f"""<s>[INST] <<SYS>>
{instruction}
<</SYS>>

{query} [/INST]"""

    response = llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repeat_penalty=repeat_penalty,
        stop=["</s>", "[INST]"],  # stronger stop signals
        echo=False,
    )

    text = response["choices"][0]["text"].strip()

    # Basic cleanup (fix escaped newlines)
    text = text.replace("\\n", "\n")

    return text

In [47]:
generate_llama_response_temparature('What is the protocol for managing sepsis in a critical care unit?')

Llama.generate: 61 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     488.89 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   27751.13 ms /   525 runs   (   52.86 ms per token,    18.92 tokens per second)
llama_perf_context_print:       total time =   28515.77 ms /   526 tokens
llama_perf_context_print:    graphs reused =        507


'1. Key Points:\n   a. Early recognition and prompt treatment are crucial in managing sepsis in a critical care unit.\n   b. Implementing the Surviving Sepsis Campaign guidelines is recommended for sepsis management.\n   c. Initial resuscitation includes administering oxygen, intravenous fluids, and antibiotics within 1 hour of recognition.\n   d. Goals are to maintain adequate tissue perfusion, prevent organ dysfunction, and provide supportive care.\n\n2. Details:\n   a. Recognition: Identify sepsis through clinical suspicion based on signs of infection, organ dysfunction, and inflammation.\n   b. Early goal-directed therapy (EGDT): Administer oxygen to maintain SpO2 > 96%, administer intravenous fluids to achieve a CVP of 8-12 mmHg or MAP > 65 mmHg, and administer broad-spectrum antibiotics within 1 hour.\n   c. Source control: Address the source of infection as soon as possible through surgical intervention if necessary.\n   d. Vasopressors: Use vasopressors to maintain MAP > 65 mmH

### Observation on Query 1


**Comparison of Responses Across Temperature Settings**

Answer 1 - generated with a temperature of 0.7, is more comprehensive and clinically nuanced. It presents a clear, step-by-step protocol and incorporates specific clinical thresholds (e.g., MAP ≥ 65 mmHg, urine output ≥ 0.5 ml/kg/h), making the response both actionable and aligned with real-world practice. The higher temperature enables a broader exploration of relevant tokens, resulting in richer, more context-aware content.

In contrast Answer 2, generated with a temperature of 0.0, is more deterministic and constrained, selecting only the most probable tokens at each step. While it maintains medical accuracy, the response is less detailed, less adaptive to the multi-part nature of the query, and ultimately truncates mid-sentence—highlighting limitations in depth and completeness.

Overall, Answer 1 is superior in terms of completeness, clinical clarity, and practical utility, making it more suitable for decision-making in a critical care context.


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [48]:
generate_llama_response('What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?')

Llama.generate: 44 prefix-match hit, remaining 36 prompt tokens to eval
llama_perf_context_print:        load time =     488.89 ms
llama_perf_context_print: prompt eval time =      86.52 ms /    36 tokens (    2.40 ms per token,   416.09 tokens per second)
llama_perf_context_print:        eval time =   32559.93 ms /   600 runs   (   54.27 ms per token,    18.43 tokens per second)
llama_perf_context_print:       total time =   33642.93 ms /   636 tokens
llama_perf_context_print:    graphs reused =        580


"1. Key Points:\n- Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch located in the lower right abdomen.\n- Common symptoms include: sudden pain in the lower right abdomen, loss of appetite, nausea and vomiting, fever, and constipation or diarrhea.\n- Appendicitis cannot be cured with medicine alone; it typically requires surgical intervention to remove the appendix (appendectomy).\n\n2. Details:\nThe appendix is a small, tube-shaped structure that extends from the large intestine. Its function is not fully understood, but some believe it may play a role in the immune system. Appendicitis occurs when the appendix becomes inflamed, usually due to a blockage in its lumen. This can lead to bacterial growth and infection, which if left untreated, can result in rupture of the appendix and peritonitis, a potentially life-threatening condition.\n\nThe symptoms of appendicitis can vary from person to person, but common signs include:\n- Sudden ons

In [26]:
# Modified Parameters - Top_p = 0.85
def generate_llama_response_topp(
    query,
    instruction = """
Answer in sections:
1. Key Points
2. Details
3. Conclusion
Keep it concise.
""",
    max_tokens=1024,
    temperature=0.4,
    top_p=0.85,
    top_k=40,
    repeat_penalty=1.1,
):
    # Proper LLaMA instruction format
    prompt = f"""<s>[INST] <<SYS>>
{instruction}
<</SYS>>

{query} [/INST]"""

    response = llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repeat_penalty=repeat_penalty,
        stop=["</s>", "[INST]"],  # stronger stop signals
        echo=False,
    )

    text = response["choices"][0]["text"].strip()

    # Basic cleanup (fix escaped newlines)
    text = text.replace("\\n", "\n")

    return text

In [49]:
generate_llama_response_topp('What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?')

Llama.generate: 79 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     488.89 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   22334.40 ms /   442 runs   (   50.53 ms per token,    19.79 tokens per second)
llama_perf_context_print:       total time =   22951.52 ms /   443 tokens
llama_perf_context_print:    graphs reused =        427


"1. Key Points:\n- Appendicitis is a medical condition characterized by inflammation of the appendix, a small pouch attached to the large intestine.\n- Common symptoms include: sudden pain in the lower right abdomen, loss of appetite, nausea and vomiting, fever, and constipation or diarrhea.\n- Appendicitis is typically treated with surgery (appendectomy) due to the risk of rupture and potential complications.\n\n2. Details:\nThe appendix is a small, worm-like structure located in the lower right side of the abdomen. Its function is not clear, but it's believed to play a role in the immune system. Appendicitis occurs when the appendix becomes inflamed and swells, often due to a blockage in its lumen.\n\nThe symptoms of appendicitis can vary from person to person, but common signs include:\n- Sudden pain in the lower right abdomen that may begin as mild and gradually worsens over hours\n- Loss of appetite\n- Nausea and vomiting\n- Fever, which may be low-grade at first but can rise as h

### Observation on Query 2
Answer 1 - generated with a `top_p` of 0.95, allows the model to sample from a broader distribution of tokens, resulting in more diverse and expansive output. While the response is informative, it tends to be somewhat general and less tightly structured.

In contrast, Answer 2, generated with a lower `top_p` of 0.85, restricts sampling to a narrower set of high-probability tokens. This leads to a more focused and concise response, with improved relevance to the query.

Overall, Answer 2 performs better, as the lower `top_p` value promotes more precise and targeted content while reducing unnecessary variation.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [50]:
generate_llama_response('What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?')

Llama.generate: 46 prefix-match hit, remaining 38 prompt tokens to eval
llama_perf_context_print:        load time =     488.89 ms
llama_perf_context_print: prompt eval time =      79.33 ms /    38 tokens (    2.09 ms per token,   479.04 tokens per second)
llama_perf_context_print:        eval time =   21737.74 ms /   420 runs   (   51.76 ms per token,    19.32 tokens per second)
llama_perf_context_print:       total time =   22401.86 ms /   458 tokens
llama_perf_context_print:    graphs reused =        406


"1. Key Points:\n- Sudden patchy hair loss, also known as alopecia areata, is an autoimmune condition that causes hair loss in small patches on the scalp or other areas of the body.\n- Treatment options for alopecia areata include topical treatments, oral medications, and corticosteroid injections.\n- Lifestyle modifications such as reducing stress, improving diet, and avoiding tight hairstyles can also help manage symptoms.\n\n2. Details:\n- Topical treatments for alopecia areata include minoxidil (Rogaine) or corticosteroid creams and ointments. Minoxidil helps to stimulate hair growth, while corticosteroids reduce inflammation and suppress the immune system.\n- Oral medications such as corticosteroids, immunosuppressants, and biologics may be prescribed for more extensive hair loss or when topical treatments are not effective.\n- Corticosteroid injections directly into the bald spots can also be an effective treatment option, especially for small areas of hair loss.\n- Lifestyle mod

In [29]:
# Modified Parameters - Top_k = 80
def generate_llama_response_topk(
    query,
    instruction = """
Answer in sections:
1. Key Points
2. Details
3. Conclusion
Keep it concise.
""",
    max_tokens=1024,
    temperature=0.4,
    top_p=0.9,
    top_k=80,
    repeat_penalty=1.1,
):
    # Proper LLaMA instruction format
    prompt = f"""<s>[INST] <<SYS>>
{instruction}
<</SYS>>

{query} [/INST]"""

    response = llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repeat_penalty=repeat_penalty,
        stop=["</s>", "[INST]"],  # stronger stop signals
        echo=False,
    )

    text = response["choices"][0]["text"].strip()

    # Basic cleanup (fix escaped newlines)
    text = text.replace("\\n", "\n")

    return text

In [51]:
generate_llama_response_topk('What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?')

Llama.generate: 83 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     488.89 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   34664.50 ms /   662 runs   (   52.36 ms per token,    19.10 tokens per second)
llama_perf_context_print:       total time =   35735.84 ms /   663 tokens
llama_perf_context_print:    graphs reused =        640


'1. Key Points:\n- Sudden patchy hair loss, also known as alopecia areata, is an autoimmune condition that causes hair loss in small patches on the scalp or other areas of the body.\n- Treatment options for sudden patchy hair loss include topical treatments, oral medications, and lifestyle modifications.\n- Topical treatments include corticosteroids, minoxidil, and anthralin. Oral medications include immunosuppressants and biologics.\n- Lifestyle modifications such as reducing stress, getting enough nutrients, and avoiding tight hairstyles can help prevent or improve hair loss.\n\n2. Details:\nSudden patchy hair loss, also known as alopecia areata, is a common condition that affects both men and women. It is an autoimmune condition that causes the immune system to attack the hair follicles, leading to hair loss in small patches on the scalp or other areas of the body.\n\nThe exact cause of alopecia areata is not known, but it is believed to be related to genetics and environmental fact

### Observation on Query 3

Answer 1, generated with a `top_k` of 50, provides a broad overview by outlining multiple causes of patchy hair loss and briefly touching on associated treatment options. While informative, the response remains somewhat high-level and generalized.

In contrast, Answer 2, generated with a higher `top_k` of 80, explores more specific treatment approaches for distinct conditions such as Alopecia Areata and Traction Alopecia. This results in greater clinical depth and more detailed therapeutic guidance.

Overall, Answer 2 is more effective in this context, as it delivers deeper medical insight and more actionable recommendations.


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
generate_llama_response('What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?')

Llama.generate: 44 prefix-match hit, remaining 32 prompt tokens to eval
llama_perf_context_print:        load time =     488.89 ms
llama_perf_context_print: prompt eval time =      76.40 ms /    32 tokens (    2.39 ms per token,   418.84 tokens per second)
llama_perf_context_print:        eval time =   14379.44 ms /   275 runs   (   52.29 ms per token,    19.12 tokens per second)
llama_perf_context_print:       total time =   14780.64 ms /   307 tokens
llama_perf_context_print:    graphs reused =        266


'1. Key Points:\n- The treatment for a brain injury depends on the severity and location of the injury.\n- Immediate medical attention is crucial to prevent further damage.\n- Rehabilitation therapy is essential for recovery and improving function.\n- Medications may be prescribed to manage symptoms or prevent complications.\n\n2. Details:\n- For mild brain injuries, rest, hydration, and over-the-counter pain relievers may be sufficient.\n- Severe brain injuries may require hospitalization, surgery, and intensive care.\n- Rehabilitation therapy can include physical, occupational, speech, and cognitive rehabilitation.\n- Medications may include anti-seizure drugs, steroids, antidepressants, and stimulants.\n- Alternative therapies such as acupuncture, massage, and biofeedback may also be beneficial.\n\n3. Conclusion:\nA brain injury can result in various impairments, and treatment should be individualized based on the severity and location of the injury. Immediate medical attention is c

In [32]:
# Modified Parameters - Max_tokens = 512
def generate_llama_response_tokens(
    query,
    instruction = """
Answer in sections:
1. Key Points
2. Details
3. Conclusion
Keep it concise.
""",
    max_tokens=512,
    temperature=0.4,
    top_p=0.9,
    top_k=40,
    repeat_penalty=1.1,
):
    # Proper LLaMA instruction format
    prompt = f"""<s>[INST] <<SYS>>
{instruction}
<</SYS>>

{query} [/INST]"""

    response = llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repeat_penalty=repeat_penalty,
        stop=["</s>", "[INST]"],  # stronger stop signals
        echo=False,
    )

    text = response["choices"][0]["text"].strip()

    # Basic cleanup (fix escaped newlines)
    text = text.replace("\\n", "\n")

    return text

In [ ]:
generate_llama_response_tokens('What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?')

Llama.generate: 75 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     488.89 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   21320.32 ms /   407 runs   (   52.38 ms per token,    19.09 tokens per second)
llama_perf_context_print:       total time =   21874.51 ms /   408 tokens
llama_perf_context_print:    graphs reused =        393


'1. Key Points:\n- The treatment for a brain injury depends on the severity and location of the injury.\n- Immediate medical attention is essential to prevent further damage and promote recovery.\n- Rehabilitation therapy, such as physical, occupational, and speech therapy, can help improve function and reduce symptoms.\n- Medications may be prescribed to manage symptoms and prevent complications.\n- Lifestyle modifications, such as getting enough rest, eating a healthy diet, and avoiding alcohol and drugs, can support recovery.\n\n2. Details:\n- For mild brain injuries, also known as concussions, rest is the most important treatment. The person should avoid physical activity and mental stimulation for a few days to allow the brain to heal.\n- For moderate to severe brain injuries, emergency medical care is necessary to address any life-threatening conditions, such as bleeding or swelling in the brain. This may involve surgery, intensive care, and medications to manage symptoms and pre

### Observation on Query 4


* Answer 1, generated with `max_tokens = 1024`, delivers a comprehensive and well-rounded overview of brain injury treatment. It covers a broad spectrum of interventions, including medications, surgical options, rehabilitation, and supportive care, while addressing physical, cognitive, and emotional aspects of recovery.

* In contrast, Answer 2, generated with `max_tokens = 512`, is more concise and focused on key areas such as initial care, rehabilitation, medications, and selective surgical interventions (e.g., for hematomas or clots). While relevant, it lacks the depth and breadth seen in Answer 1.

* Overall, Answer 1 is superior, as the higher token limit enables a more detailed and holistic response, ensuring more complete coverage of the topic.


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
generate_llama_response('What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?')

Llama.generate: 44 prefix-match hit, remaining 39 prompt tokens to eval
llama_perf_context_print:        load time =     488.89 ms
llama_perf_context_print: prompt eval time =      83.73 ms /    39 tokens (    2.15 ms per token,   465.81 tokens per second)
llama_perf_context_print:        eval time =   23477.44 ms /   456 runs   (   51.49 ms per token,    19.42 tokens per second)
llama_perf_context_print:       total time =   24217.97 ms /   495 tokens
llama_perf_context_print:    graphs reused =        441


"1. Key Points:\n- Ensure the safety of the injured person and prevent further injury.\n- Assess the severity of the fracture and provide initial first aid.\n- Immediately immobilize the leg using a splint or sling to minimize movement and swelling.\n- Seek medical attention as soon as possible.\n- Provide pain relief with over-the-counter medications, such as acetaminophen or ibuprofen.\n- Elevate the injured leg above heart level to reduce swelling and promote healing.\n- Encourage proper nutrition and hydration for optimal recovery.\n\n2. Details:\n- Safety first: Ensure the safety of both the injured person and any companions by securing the area or calling for help if necessary.\n- Assess the severity: Check for signs of open fractures, deformities, or numbness in the leg. If the fracture is suspected to be severe, do not move the person unless it is necessary for their safety.\n- First aid: Apply a clean pad over the injury and secure it with an elastic bandage or a splint. Do no

In [35]:
# Modified Parameters - Temp = 0.7,Top_p = 0.85,Top_k = 80,Max_tokens = 1024
def generate_llama_response_modified(
    query,
    instruction = """
Answer in sections:
1. Key Points
2. Details
3. Conclusion
Keep it concise.
""",
    max_tokens=1024,
    temperature=0.7,
    top_p=0.85,
    top_k=80,
    repeat_penalty=1.1,
):
    # Proper LLaMA instruction format
    prompt = f"""<s>[INST] <<SYS>>
{instruction}
<</SYS>>

{query} [/INST]"""

    response = llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k,
        repeat_penalty=repeat_penalty,
        stop=["</s>", "[INST]"],  # stronger stop signals
        echo=False,
    )

    text = response["choices"][0]["text"].strip()

    # Basic cleanup (fix escaped newlines)
    text = text.replace("\\n", "\n")

    return text

In [ ]:
generate_llama_response_modified('What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?')

Llama.generate: 82 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =     488.89 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =   21451.21 ms /   414 runs   (   51.81 ms per token,    19.30 tokens per second)
llama_perf_context_print:       total time =   22009.36 ms /   415 tokens
llama_perf_context_print:    graphs reused =        400


"1. Key Points:\n- Secure the area to prevent further injury or movement of the fractured leg. Use a splint, sling, or immobilizing device if available.\n- Apply ice to reduce swelling and pain.\n- Elevate the injured leg above heart level to minimize swelling and discomfort.\n- Seek medical attention as soon as possible.\n- Administer pain relief medication as prescribed by a healthcare professional.\n\n2. Details:\n- To secure the fractured leg, use a splint or sling to keep it stable and prevent any weight from being put on it. If no such device is available, try to immobilize the leg using items like sticks, branches, or clothing.\n- Apply ice packs to the injury for 15-20 minutes every hour for the first few days to help reduce swelling and pain. Do not apply ice directly to the skin; wrap it in a cloth or protective layer first.\n- Elevate the injured leg above heart level whenever possible to minimize swelling and discomfort. Propping up pillows under the affected limb will help

### Observation on Query 5

* Answer 1, generated with default parameters, outlines the immediate steps for managing a fractured leg during a hiking trip. It emphasizes initial assessment, basic stabilization, infection prevention, and the importance of seeking medical attention in the event of complications.

* In contrast, Answer 2, generated with modified parameters, is more structured and comprehensive. It not only addresses immediate first-aid measures but also incorporates longer-term considerations such as rest, nutrition, and physical therapy. Additionally, it focuses on practical, actionable steps a layperson can take in a remote setting, such as wound care and leg elevation for pain management.

* Overall, Answer 2 is more effective, as it provides a more holistic view by covering both immediate response and ongoing recovery, making it better suited for end-to-end guidance.


## Data Preparation for RAG

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

### Loading the Data

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
merck_pdf_path = '/content/drive/MyDrive/Notebooks/NLP/Assignment/medical_diagnosis_manual.pdf'
pdf_loader = PyMuPDFLoader(merck_pdf_path)
merck = pdf_loader.load()
merck = merck[2:]  # skip first 2 pages


### Data Overview

#### Checking the first 5 pages

In [7]:
# Code to print first 5 pages
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(merck[i].page_content,end="\n")

Page Number : 1
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ...........................................................................................................................................................................................
53
1 - Nutritional Disorders    ...............................................................................................................................................................
53
Chapter 1. Nutrition: General Considerations    ......................................................................................

#### Checking the number of pages

In [8]:
# Code to check number of Pages
len(merck)

4112

### Data Chunking

In [9]:
# Initializing a RecursiveCharacterTextSplitter to split the text into manageable chunks for embedding and retrieval
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap= 20
)

# Load and split documents
document_chunks = pdf_loader.load_and_split(text_splitter)


In [10]:
# Basic inspection
print(f"Total chunks created: {len(document_chunks)}\n")

# Sample chunks for validation
for i in [2,3,4]:
    print(f"--- Chunk {i} ---")
    print(document_chunks[i].page_content[:500])  # print first 500 chars
    print("\n")

Total chunks created: 8492

--- Chunk 2 ---
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    .................................


--- Chunk 3 ---
Chapter 24. Testing for Hepatic & Biliary Disorders    ......................................................................................................
305
Chapter 25. Drugs & the Liver    ................................................................................................................................................
308
Chapter 26. Alcoholic Liver Disease    ......................................................

### Embedding

In [17]:
# Upgrade SciPy and Scikit-Learn to NumPy 2.x compatible versions
!pip install --upgrade "scipy>=1.13.0" "scikit-learn>=1.5.0" -q



In [19]:
# Upgrade Sentence Transformers to support NumPy 2.x
!pip install --upgrade sentence-transformers>=3.0.0 -q

In [3]:
#This model is chosen because of its embedding vector size is the same as our token size in chunking (512).
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')

/tmp/ipykernel_9537/487786534.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or dataset

modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [11]:
# Generating embedding for the first document chunk
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
# Generating embedding for the second document chunk
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [12]:
#Checking if both are of the same size
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

### Vector Database

In [13]:
import os

In [14]:
# Creating the output directory 'merck_db' if it doesn't already exist, so we can save the processed data or vector database files there.
out_dir = 'merck_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [15]:
vectorstore = Chroma.from_documents( #creating a Chroma vector store from a set of document chunks.
    document_chunks, #creating a list of text chunks that will be converted into embeddings..
    embedding_model, #model responsible for embedding the document chunks into vector representations
    persist_directory=out_dir #name of the collection in the Chroma database
)

In [16]:
 #Loading Chroma vector store with the given embedding model
 vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

/tmp/ipykernel_9537/382498760.py:2: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)


In [17]:
#Accessing the embedding function used in the Chroma vector store
vectorstore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

### Retriever

In [18]:
#Performing a similarity search in the vector store to find the top 3 most similar documents to "Alopecia Areata"
vectorstore.similarity_search("Alopecia Areata ",k=3)

[Document(metadata={'creationdate': '2012-06-15T05:44:40+00:00', 'subject': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'source': '/content/drive/MyDrive/Notebooks/NLP/Assignment/medical_diagnosis_manual.pdf', 'file_path': '/content/drive/MyDrive/Notebooks/NLP/Assignment/medical_diagnosis_manual.pdf', 'moddate': '2026-03-10T14:39:40+00:00', 'creator': 'Atop CHM to PDF Converter', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'author': '', 'page': 859, 'creationDate': 'D:20120615054440Z', 'format': 'PDF 1.7', 'trapped': '', 'modDate': 'D:20260310143940Z', 'total_pages': 4114, 'keywords': ''}, page_content='infection or emotional stress. It occasionally coexists with autoimmune vitiligo or thyroiditis.\nDiagnosis\n• Examination\nDiagnosis is by inspection. Alopecia areata typically manifests as discrete circular patches of hair loss\ncharacterized by short broken hairs at the margins, which resemble exclamation points. Nails are\nsometimes pitted o

### Observations

From the retrieved chunks, we observe that all the chunks are related to the key terms Alopecia Areata

In [19]:
retriever = vectorstore.as_retriever( #Converting the Chroma vector store into a retriever for querying.
    search_type='similarity', #Specifying that retrieval is based on cosine similarity
    search_kwargs={'k': 3} #Retrieving the top 3 most similar documents for a given query.
)

In [20]:
user_input = 'What are the symptoms of migraine?'
rel_docs = retriever.get_relevant_documents(user_input)
rel_docs

/tmp/ipykernel_9537/2691306525.py:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  rel_docs = retriever.get_relevant_documents(user_input)


[Document(metadata={'author': '', 'keywords': '', 'file_path': '/content/drive/MyDrive/Notebooks/NLP/Assignment/medical_diagnosis_manual.pdf', 'modDate': 'D:20260310143940Z', 'format': 'PDF 1.7', 'creationdate': '2012-06-15T05:44:40+00:00', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'moddate': '2026-03-10T14:39:40+00:00', 'trapped': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'total_pages': 4114, 'source': '/content/drive/MyDrive/Notebooks/NLP/Assignment/medical_diagnosis_manual.pdf', 'creationDate': 'D:20120615054440Z', 'page': 1895, 'subject': '', 'creator': 'Atop CHM to PDF Converter'}, page_content='are less common than visual auras. Some patients have an aura with little or no headache.\nHeadache varies from moderate to severe, and attacks last from 4 hours to several days, typically\nresolving with sleep. The pain is often unilateral but may be bilateral, most often in a frontotemporal\ndistribution, and is typically described as pulsati

### System and User Prompt Template

In [21]:
# System message instructing the LLM to only answer using Merck Manual 19th Edition
qna_system_message = """
You are a helpful assistant trained to answer questions based only on the Merck Manual of Medical Diagnosis and Therapy, Nineteenth Edition.
Use the context provided to find accurate and reliable answers.
If the answer is not found in the context, reply with "I don't know".
Do not mention the context or the Merck Manual in your final answer.
"""

In [22]:
# Template for formatting the user's input with context from the Merck Manual, 19th Edition and the actual medical question.
qna_user_message_template = """
###Context
The following excerpts are from the Merck Manual of Medical Diagnosis and Therapy, Nineteenth Edition:
{context}

###Question
{question}
"""

### Response Function

In [32]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [33]:
user_input1 = 'What is the protocol for managing sepsis in a critical care unit?'
print(generate_rag_response(user_input1))

llama_perf_context_print:        load time =    2900.58 ms
llama_perf_context_print: prompt eval time =    2900.10 ms /  1818 tokens (    1.60 ms per token,   626.87 tokens per second)
llama_perf_context_print:        eval time =    6907.66 ms /   127 runs   (   54.39 ms per token,    18.39 tokens per second)
llama_perf_context_print:       total time =    9879.81 ms /  1945 tokens
llama_perf_context_print:    graphs reused =        122


###Answer
The protocol for managing sepsis in a critical care unit includes prompt empiric antibiotic therapy, source control such as drainage of abscesses or surgical excision of necrotic tissues, normalization of blood glucose, corticosteroid therapy, and supportive care. Antibiotics should be based on the suspected source and causative organisms, and adjusted based on culture and sensitivity results. Gentamicin or tobramycin and a 3rd-generation cephalosporin, or ceftazidime plus a fluoro


In [ ]:
# Fine Tuning - Removing Chunk Overlap
# Chunk size = 512, overlap = 0

In [34]:
# Initializing a RecursiveCharacterTextSplitter to split the text into manageable chunks for embedding and retrieval
text_splitter1 = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap= 0
)

In [35]:
#loading the PDF document, extracting its text, and splitting it into smaller chunks
document_chunks1 = pdf_loader.load_and_split(text_splitter1)

In [36]:
#Checking the number of text chunks the pdf has been split into
len(document_chunks1)

8466

In [37]:
# Generating embedding for the first document chunk
embedding_1 = embedding_model.embed_query(document_chunks1[0].page_content)
# Generating embedding for the second document chunk
embedding_2 = embedding_model.embed_query(document_chunks1[1].page_content)

In [38]:
vectorstore1 = Chroma.from_documents( #creating a Chroma vector store from a set of document chunks.
    document_chunks1, #creating a list of text chunks that will be converted into embeddings..
    embedding_model, #model responsible for embedding the document chunks into vector representations
    persist_directory=out_dir #name of the collection in the Chroma database
)

In [40]:
 #Loading Chroma vector store with the given embedding model
 vectorstore1 = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

In [41]:
#Accessing the embedding function used in the Chroma vector store
vectorstore1.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

In [42]:
retriever1 = vectorstore1.as_retriever( #Converting the Chroma vector store into a retriever for querying.
    search_type='similarity', #Specifying that retrieval is based on cosine similarity
    search_kwargs={'k': 3} #Retrieving the top 3 most similar documents for a given query.
)

In [43]:
def generate_rag_response1(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks1 = retriever1.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks1]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

In [44]:
user_input1 = 'What is the protocol for managing sepsis in a critical care unit?'
print(generate_rag_response1(user_input1))

Llama.generate: 116 prefix-match hit, remaining 1742 prompt tokens to eval
llama_perf_context_print:        load time =    2900.58 ms
llama_perf_context_print: prompt eval time =    2451.26 ms /  1742 tokens (    1.41 ms per token,   710.66 tokens per second)
llama_perf_context_print:        eval time =    6864.18 ms /   127 runs   (   54.05 ms per token,    18.50 tokens per second)
llama_perf_context_print:       total time =    9399.04 ms /  1869 tokens
llama_perf_context_print:    graphs reused =        122


###Answer
The protocol for managing sepsis in a critical care unit includes giving parenteral antibiotics after taking specimens for Gram stain and culture, starting empiric therapy immediately, and selecting antibiotics based on the suspected source, clinical setting, and previous culture results. One regimen for septic shock of unknown cause is gentamicin or tobramycin 5.1 mg/kg IV once/day plus a 3rd-generation cephalosporin or ceftazidime plus a fluoroquinolone. Vancomycin should be added



### **Observations on Query 1**

Removing chunk overlap did not impact the response in this case, likely because all relevant information needed to answer the query was contained within a single chunk.

However, chunk overlap becomes important when the required context spans multiple chunks. In such scenarios, removing overlap can lead to fragmented context and potentially incomplete or less accurate answers.


### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [45]:
# Basic Parameters
user_input2 = 'What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?'
print(generate_rag_response(user_input2))

Llama.generate: 116 prefix-match hit, remaining 1539 prompt tokens to eval
llama_perf_context_print:        load time =    2900.58 ms
llama_perf_context_print: prompt eval time =    2193.89 ms /  1539 tokens (    1.43 ms per token,   701.49 tokens per second)
llama_perf_context_print:        eval time =    7187.51 ms /   127 runs   (   56.59 ms per token,    17.67 tokens per second)
llama_perf_context_print:       total time =    9462.32 ms /  1666 tokens
llama_perf_context_print:    graphs reused =        122


The common symptoms for appendicitis include abdominal pain, anorexia, and abdominal tenderness. Appendicitis cannot be cured via medicine alone, and the treatment is surgical removal of the appendix. This procedure is called appendectomy. It can be performed either open or laparoscopically. Antibiotics are also administered before and after the surgery to prevent infection. If the appendix is perforated, antibiotics should be continued until the patient's temperature and WBC count have normalized. In cases where a large inflammatory mass is found involving


In [46]:
# Fine Tuning - Retreiver Parameters
# k = 2
retriever2 = vectorstore.as_retriever( #Converting the Chroma vector store into a retriever for querying.
    search_type='similarity', #Specifying that retrieval is based on cosine similarity
    search_kwargs={'k': 2} #Retrieving the top 2 most similar documents for a given query.
)

In [51]:
def generate_rag_response2(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever2.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

In [50]:
user_input2 = 'What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?'
print(generate_rag_response2(user_input2))

Llama.generate: 1007 prefix-match hit, remaining 39 prompt tokens to eval
llama_perf_context_print:        load time =    2900.58 ms
llama_perf_context_print: prompt eval time =     139.00 ms /    39 tokens (    3.56 ms per token,   280.58 tokens per second)
llama_perf_context_print:        eval time =    7196.86 ms /   127 runs   (   56.67 ms per token,    17.65 tokens per second)
llama_perf_context_print:       total time =    7415.17 ms /   166 tokens
llama_perf_context_print:    graphs reused =        122


The common symptoms for appendicitis include abdominal pain, anorexia, and abdominal tenderness. Appendicitis cannot be cured via medicine, and the treatment is surgical removal of the appendix. This procedure is called appendectomy. It can be performed either open or laparoscopically. Antibiotics are also administered before the surgery to prevent infection. If the appendix is perforated, antibiotics should be continued until the patient's temperature and WBC count have normalized. In cases where a large inflammatory mass is found involving the appendix


## **Observations on Query 2**

* **Answer 1 (k = 3):**

  * More detailed and comprehensive
  * Includes additional context and examples (e.g., scenarios where surgery is not feasible)
  * Benefits from retrieving more relevant chunks

* **Answer 2 (k = 2):**

  * Shorter and less detailed
  * Cuts off earlier, indicating limited context
  * Fewer retrieved chunks likely reduced completeness

* **Overall Insight:**

  * Increasing `k` (number of retrieved chunks) improves context richness and leads to more complete, informative responses


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [52]:
# Basic parameters
user_input3 = 'What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?'
print(generate_rag_response(user_input3))

Llama.generate: 116 prefix-match hit, remaining 1373 prompt tokens to eval
llama_perf_context_print:        load time =    2900.58 ms
llama_perf_context_print: prompt eval time =    2042.58 ms /  1373 tokens (    1.49 ms per token,   672.19 tokens per second)
llama_perf_context_print:        eval time =    7185.58 ms /   127 runs   (   56.58 ms per token,    17.67 tokens per second)
llama_perf_context_print:       total time =    9305.80 ms /  1500 tokens
llama_perf_context_print:    graphs reused =        122


Based on the context provided, sudden patchy hair loss, also known as alopecia areata, can be treated with various options including topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). Hormonal modulators such as oral contraceptives or spironolactone may be useful for female-pattern hair loss associated with hyperandrogen


In [54]:
#Fine Tuning - LLM Parameters - Temperature = 0.7
user_input3 = 'What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?'
print(generate_rag_response(user_input3, temperature=0.7))

Llama.generate: 1488 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    2900.58 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =    7253.16 ms /   128 runs   (   56.67 ms per token,    17.65 tokens per second)
llama_perf_context_print:       total time =    7335.03 ms /   129 tokens
llama_perf_context_print:    graphs reused =        123


The Merck Manual mentions that sudden patchy hair loss, also known as alopecia areata, is a condition in which people experience hair loss in patches with no obvious skin or systemic disorder. The scalp and beard are most frequently affected, but any hairy area may be involved. It is an autoimmune disorder affecting genetically susceptible people exposed to unclear environmental triggers.

The treatment options for alopecia areata include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, top


## **Observations on Query 3**

* **Answer 1 (temperature = 0):**

  * More precise and deterministic
  * Consistent and well-structured output
  * Focuses on the most probable and direct response

* **Answer 2 (temperature = 0.7):**

  * Introduces greater variability in phrasing
  * Slightly more dynamic and expressive
  * Reflects increased randomness due to higher temperature

* **Overall Insight:**

  * Lower temperature improves consistency and precision, while higher temperature adds variation at the cost of determinism


### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [56]:
#Basic paramateres
user_input4 = 'What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?'
print(generate_rag_response(user_input4))

Llama.generate: 116 prefix-match hit, remaining 1493 prompt tokens to eval
llama_perf_context_print:        load time =    2900.58 ms
llama_perf_context_print: prompt eval time =    2107.96 ms /  1493 tokens (    1.41 ms per token,   708.27 tokens per second)
llama_perf_context_print:        eval time =    6670.29 ms /   116 runs   (   57.50 ms per token,    17.39 tokens per second)
llama_perf_context_print:       total time =    8857.90 ms /  1609 tokens
llama_perf_context_print:    graphs reused =        112


###Answer
The person should receive supportive care, which includes preventing systemic complications, providing good nutrition, and preventing pressure ulcers. There is no specific treatment for the brain injury itself. In severe cases, surgery may be needed to place monitors to track and treat intracranial pressure, decompress the brain, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabilitation.


In [57]:
#Fine Tuning - LLM Parameters Top_p = 0.8
user_input4 = 'What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?'
print(generate_rag_response(user_input4, top_p=0.8))

Llama.generate: 1608 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    2900.58 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =    6422.60 ms /   117 runs   (   54.89 ms per token,    18.22 tokens per second)
llama_perf_context_print:       total time =    6499.19 ms /   118 tokens
llama_perf_context_print:    graphs reused =        113


###Answer
The person should receive supportive care, which includes preventing systemic complications, providing good nutrition, and preventing pressure ulcers. There is no specific treatment for the brain injury itself. In severe cases, surgery may be needed to place monitors to track and treat intracranial pressure, decompress the brain, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabilitation.


## **Observations on Query 4**

* **Answer 1 (top_p = 0.95):**

  * Samples from a broader probability mass (top 95%)
  * Produces more coherent and well-structured responses
  * Stays focused on core concepts (e.g., supportive care, rehabilitation)
  * Minimizes inclusion of less relevant details

* **Answer 2 (top_p = 0.8):**

  * Restricts sampling to a narrower probability mass (top 80%)
  * Results in more selective but slightly varied outputs
  * Introduces additional perspectives (e.g., severe cases, surgical interventions, treatment phases)

* **Overall Insight:**

  * Higher `top_p` (0.95) leads to more stable and focused responses, while lower `top_p` (0.8) can introduce variation and expand coverage, sometimes adding extra but less central details


### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [59]:
#Basic Parameteres
user_input5 = 'What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?'
print(generate_rag_response(user_input5))

Llama.generate: 116 prefix-match hit, remaining 1441 prompt tokens to eval
llama_perf_context_print:        load time =    2900.58 ms
llama_perf_context_print: prompt eval time =    2090.33 ms /  1441 tokens (    1.45 ms per token,   689.36 tokens per second)
llama_perf_context_print:        eval time =    8036.85 ms /   127 runs   (   63.28 ms per token,    15.80 tokens per second)
llama_perf_context_print:       total time =   10231.43 ms /  1568 tokens
llama_perf_context_print:    graphs reused =        122


Based on the context provided, the following are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip:

1. Elevate the injured limb above heart level for the first 2 days to minimize swelling and allow gravity to help drain edema fluid.
2. Apply elastic bandages (batting, and layers 2 and 4) to the injured limb.
3. After 48 hours, apply periodic warmth (eg, a heating pad) for 15 to 20 minutes to relieve pain and speed healing


In [60]:
#Fine Tuning - LLM Parameters Top_k = 25
user_input5 = 'What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?'
print(generate_rag_response(user_input5, top_k=25))

Llama.generate: 1556 prefix-match hit, remaining 1 prompt tokens to eval
llama_perf_context_print:        load time =    2900.58 ms
llama_perf_context_print: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_perf_context_print:        eval time =    7224.29 ms /   128 runs   (   56.44 ms per token,    17.72 tokens per second)
llama_perf_context_print:       total time =    7297.47 ms /   129 tokens
llama_perf_context_print:    graphs reused =        123


Based on the context provided, the following are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip:

1. Elevate the injured limb above heart level for the first 2 days to minimize swelling and allow gravity to help drain edema fluid.
2. Apply elastic bandages (batting, and layers 2 and 4) to the injured limb.
3. After 48 hours, apply periodic warmth (eg, a heating pad) for 15 to 20 minutes to relieve pain and speed healing


### Observations

* In this case, both responses are identical, likely because the topic is highly structured and deterministic. Even with a broader candidate pool (`top_k = 50`), the model converges on the same high-probability tokens, resulting in consistent outputs.


## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [61]:
groundedness_rater_system_message = """

You will be given:
- a ###Question
- the ###Context used by the AI system
- the AI-generated ###Answer

Your task is to evaluate how well the ###Answer is grounded in the provided ###Context.

Rate the groundedness on a scale of 1 to 5:

1 – The answer is not derived from the context at all
2 – The answer is minimally derived from the context
3 – The answer is partially derived from the context
4 – The answer is mostly derived from the context
5 – The answer is fully derived from the context

Instructions:
- Provide a single overall rating (1–5)
- Include a brief explanation justifying your rating
- Base your judgment strictly on the given context (do not use external knowledge)

"""


In [62]:
relevance_rater_system_message = """

You will be given:
- a ###Question
- the ###Context used by the AI system
- the AI-generated ###Answer

Your task is to evaluate how relevant the ###Answer is to the ###Question, considering whether it directly addresses the key aspects of the question based on the provided context.

Rate the relevance on a scale of 1 to 5:

1 – The answer is not relevant to the question at all
2 – The answer is slightly relevant but misses most key aspects
3 – The answer is moderately relevant, addressing some parts but missing important details
4 – The answer is mostly relevant, covering key aspects with minor gaps
5 – The answer is fully relevant, directly and completely addressing the question using the context

Instructions:
- Provide a single overall rating (1–5)
- Include a brief explanation justifying your rating
- Base your evaluation on both the question and the provided context

"""


In [64]:
user_message_template = """

### Question
{question}

### Context
{context}

### Answer
{answer}

"""




In [65]:
def generate_ground_relevance_response(
    user_input: str,
    k: int = 3,
    max_tokens: int = 128,
    temperature: float = 0,
    top_p: float = 0.95,
    top_k: int = 50
) -> tuple[str, str]:
    """
    Generates a response grounded in retrieved documents and evaluates its
    groundedness and relevance.

    Returns:
        tuple: (groundedness_rating, relevance_rating)
    """
    global qna_system_message, qna_user_message_template

    # Retrieve top-k relevant document chunks
    relevant_docs = retriever.get_relevant_documents(query=user_input, k=k)
    context = ". ".join(doc.page_content for doc in relevant_docs)

    # Build main prompt for LLM response
    user_prompt = qna_user_message_template.format(context=context, question=user_input)
    prompt = f"[INST]{qna_system_message}\nuser: {user_prompt}[/INST]"

    def call_llm(prompt_text: str) -> str:
        """Helper to call LLM and return text output."""
        response = llm(
            prompt=prompt_text,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST']
        )
        return response['choices'][0]['text']

    # Generate LLM answer
    answer = call_llm(prompt)

    # Build groundedness and relevance prompts
    groundedness_prompt = f"[INST]{groundedness_rater_system_message}\nuser: {user_message_template.format(context=context, question=user_input, answer=answer)}[/INST]"
    relevance_prompt = f"[INST]{relevance_rater_system_message}\nuser: {user_message_template.format(context=context, question=user_input, answer=answer)}[/INST]"

    # Evaluate groundedness and relevance
    groundedness_rating = call_llm(groundedness_prompt)
    relevance_rating = call_llm(relevance_prompt)

    return groundedness_rating, relevance_rating


### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [68]:
user_input1 = 'What is the protocol for managing sepsis in a critical care unit?'
ground,rel = generate_ground_relevance_response(user_input1,max_tokens=350)

print(ground,end="\n\n")
print(rel)

llama_perf_context_print:        load time =    2594.35 ms
llama_perf_context_print: prompt eval time =    2593.93 ms /  1867 tokens (    1.39 ms per token,   719.76 tokens per second)
llama_perf_context_print:        eval time =   14901.36 ms /   273 runs   (   54.58 ms per token,    18.32 tokens per second)
llama_perf_context_print:       total time =   17704.64 ms /  2140 tokens
llama_perf_context_print:    graphs reused =        264
Llama.generate: 5 prefix-match hit, remaining 2215 prompt tokens to eval
llama_perf_context_print:        load time =    2594.35 ms
llama_perf_context_print: prompt eval time =    3182.26 ms /  2215 tokens (    1.44 ms per token,   696.05 tokens per second)
llama_perf_context_print:        eval time =    8273.04 ms /   138 runs   (   59.95 ms per token,    16.68 tokens per second)
llama_perf_context_print:       total time =   11535.17 ms /  2353 tokens
llama_perf_context_print:    graphs reused =        133
Llama.generate: 43 prefix-match hit, remainin

 Rating: 5

Explanation: The answer is fully derived from the context. The context provides detailed information about the management of sepsis in a critical care unit, including taking specimens for Gram stain and culture before giving parenteral antibiotics, starting prompt empiric therapy, selecting antibiotics based on the suspected source and previous culture results, adding Vancomycin for resistant bacteria, including a drug effective against anaerobes for abdominal sources, continuing antibiotics for at least 5 days, draining abscesses, and normalizing blood glucose. The answer summarizes this information accurately.

 Rating: 5

Explanation: The answer directly and completely addresses the question using the context provided. The context mentions the importance of managing infections in critical care units, and the question asks specifically about the protocol for managing sepsis. The answer outlines the steps for managing sepsis, including giving parenteral antibiotics, starti

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [69]:
user_input2 = 'What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?'
ground,rel = generate_ground_relevance_response(user_input2,max_tokens=350)

print(ground,end="\n\n")
print(rel)

Llama.generate: 5 prefix-match hit, remaining 1659 prompt tokens to eval
llama_perf_context_print:        load time =    2594.35 ms
llama_perf_context_print: prompt eval time =    2259.07 ms /  1659 tokens (    1.36 ms per token,   734.37 tokens per second)
llama_perf_context_print:        eval time =    3139.58 ms /    61 runs   (   51.47 ms per token,    19.43 tokens per second)
llama_perf_context_print:       total time =    5431.12 ms /  1720 tokens
llama_perf_context_print:    graphs reused =         59
Llama.generate: 5 prefix-match hit, remaining 1800 prompt tokens to eval
llama_perf_context_print:        load time =    2594.35 ms
llama_perf_context_print: prompt eval time =    2505.00 ms /  1800 tokens (    1.39 ms per token,   718.56 tokens per second)
llama_perf_context_print:        eval time =    4253.10 ms /    79 runs   (   53.84 ms per token,    18.57 tokens per second)
llama_perf_context_print:       total time =    6805.07 ms /  1879 tokens
llama_perf_context_print:   

 Rating: 5

Explanation: The context clearly states that the common symptoms for appendicitis are abdominal pain, anorexia, and abdominal tenderness. It also states that treatment for appendicitis is surgical removal of the appendix through an open or laparoscopic appendectomy. The answer is fully derived from the context.

 Rating: 5

Explanation: The answer directly and completely addresses the question by listing the common symptoms for appendicitis and stating that it cannot be cured via medicine alone, with the standard treatment being surgical removal of the appendix. The context provided in the question and the context in the text given to the AI system both support this answer.


### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [70]:
user_input3 = 'What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?'
ground,rel = generate_ground_relevance_response(user_input3,max_tokens=350)

print(ground,end="\n\n")
print(rel)

Llama.generate: 5 prefix-match hit, remaining 1493 prompt tokens to eval
llama_perf_context_print:        load time =    2594.35 ms
llama_perf_context_print: prompt eval time =    2198.54 ms /  1493 tokens (    1.47 ms per token,   679.09 tokens per second)
llama_perf_context_print:        eval time =   10423.13 ms /   184 runs   (   56.65 ms per token,    17.65 tokens per second)
llama_perf_context_print:       total time =   12740.67 ms /  1677 tokens
llama_perf_context_print:    graphs reused =        177
Llama.generate: 5 prefix-match hit, remaining 1757 prompt tokens to eval
llama_perf_context_print:        load time =    2594.35 ms
llama_perf_context_print: prompt eval time =    2635.36 ms /  1757 tokens (    1.50 ms per token,   666.70 tokens per second)
llama_perf_context_print:        eval time =    8474.22 ms /   141 runs   (   60.10 ms per token,    16.64 tokens per second)
llama_perf_context_print:       total time =   11193.17 ms /  1898 tokens
llama_perf_context_print:   

 Rating: 5

Explanation: The context clearly states that alopecia areata is a type of sudden patchy hair loss that can be treated with various options including topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). The context also mentions the possible causes of alopecia areata being autoimmune disorders and unclear environmental triggers. Therefore, the answer is fully derived from the context.

 Rating: 5

Explanation: The question asks about effective treatments and possible causes for sudden patchy hair loss, specifically alopecia areata. The context provided in the text discusses various treatments for alopecia areata, including topical, intralesional, and systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy, and psoralen plus ultraviolet A. The answer directly addresses the question by listing these same 

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [71]:
user_input4 = 'What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?'
ground,rel = generate_ground_relevance_response(user_input4,max_tokens=350)

print(ground,end="\n\n")
print(rel)

Llama.generate: 5 prefix-match hit, remaining 1613 prompt tokens to eval
llama_perf_context_print:        load time =    2594.35 ms
llama_perf_context_print: prompt eval time =    2252.92 ms /  1613 tokens (    1.40 ms per token,   715.96 tokens per second)
llama_perf_context_print:        eval time =    7546.93 ms /   143 runs   (   52.78 ms per token,    18.95 tokens per second)
llama_perf_context_print:       total time =    9905.24 ms /  1756 tokens
llama_perf_context_print:    graphs reused =        137
Llama.generate: 5 prefix-match hit, remaining 1836 prompt tokens to eval
llama_perf_context_print:        load time =    2594.35 ms
llama_perf_context_print: prompt eval time =    2530.88 ms /  1836 tokens (    1.38 ms per token,   725.44 tokens per second)
llama_perf_context_print:        eval time =    8348.12 ms /   152 runs   (   54.92 ms per token,    18.21 tokens per second)
llama_perf_context_print:       total time =   10990.82 ms /  1988 tokens
llama_perf_context_print:   

 Rating: 5

Explanation: The answer is fully derived from the context. The context describes the initial treatment for a person with a brain injury as ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood pressure, and the possibility of surgery to treat intracranial pressure, decompress the brain, or remove hematomas. The answer also mentions the importance of maintaining adequate brain perfusion and oxygenation in the first few days and the need for rehabilitation later on. The context also states that there is no specific treatment for brain injury, but supportive care is necessary to prevent systemic complications, provide good nutrition, and prevent pressure ulcers. The answer accurately reflects this information from the context.

 Rating: 5

Explanation: The answer directly addresses the key aspects of the question by mentioning the initial treatments for a person with a physical injury to brain tissue, which include ensuring a reliable airway, mai

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [72]:
user_input5 = 'What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?'
ground,rel = generate_ground_relevance_response(user_input5,max_tokens=350)

print(ground,end="\n\n")
print(rel)

Llama.generate: 5 prefix-match hit, remaining 1561 prompt tokens to eval
llama_perf_context_print:        load time =    2594.35 ms
llama_perf_context_print: prompt eval time =    2219.77 ms /  1561 tokens (    1.42 ms per token,   703.23 tokens per second)
llama_perf_context_print:        eval time =   14906.03 ms /   267 runs   (   55.83 ms per token,    17.91 tokens per second)
llama_perf_context_print:       total time =   17331.12 ms /  1828 tokens
llama_perf_context_print:    graphs reused =        257
Llama.generate: 5 prefix-match hit, remaining 1908 prompt tokens to eval
llama_perf_context_print:        load time =    2594.35 ms
llama_perf_context_print: prompt eval time =    2689.91 ms /  1908 tokens (    1.41 ms per token,   709.32 tokens per second)
llama_perf_context_print:        eval time =    5031.24 ms /    87 runs   (   57.83 ms per token,    17.29 tokens per second)
llama_perf_context_print:       total time =    7777.03 ms /  1995 tokens
llama_perf_context_print:   

 Rating: 5

Explanation: The answer is fully derived from the context. The context provides detailed information about the necessary precautions and treatment steps for a person with a fractured leg, and the answer summarizes and repeats that information. The answer also includes some additional information about the use of a splint and the potential complications of prolonged immobilization, which is also discussed in the context.

 Rating: 5

Explanation: The answer directly and completely addresses the question by outlining the necessary precautions and treatment steps for a person with a fractured leg, based on the provided context from the Merck Manual. It covers key aspects such as elevating the injured limb, applying heat, immobilization, using a cast or splint, and maintaining good hygiene. The answer also mentions the importance of seeking medical care and the potential complications of prolonged immobilization. Therefore, the answer is fully relevant to the question.


**Observations**

A rating of 5 for both groundedness and relevance indicates that the answer is completely supported by the provided context and is highly pertinent to the query.


## Actionable Insights and Business Recommendations

**Business Insights**

* **Accuracy and Trustworthiness**

  RAG-based answers are context-specific, reliable, and trustworthy, enhancing customer satisfaction in medical applications.

* **Enhanced User Experience**

  The RAG model delivers personalized, relevant responses with higher accuracy, improving the overall user experience.

* **Cost Reduction in Medical Consultations**

  Automating medical Q&A using the model can reduce the need for consultations, lowering costs in telemedicine and healthcare support.

**Recommendations**

* **Continuous Fine-tuning**

  Regular updates and fine-tuning are essential to keep the model aligned with the latest medical guidelines and research.

* **Ethical Considerations**

  Prioritize data privacy and ensure compliance with regulations such as HIPAA and GDPR.

* **Specialized Models**

  Develop dedicated models for specific medical specialties—like orthopedics, gastroenterology, oncology—for improved accuracy and relevance.